# Preprocessing XGBoost 

## Cell 1 — Setup & Konfigurasi

In [5]:
import pandas as pd
import numpy as np
import re, os, pickle, time, warnings
warnings.filterwarnings('ignore')
from functools import lru_cache

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)

# ── PATH ─────────────────────────────────────────────────────────────────────
BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
FILE_IN    = os.path.join(BASE_DIR, 'labelled_data_filtered.csv')
PREP_DIR   = os.path.join(BASE_DIR, 'prep_v2')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
for d in [PREP_DIR, MODEL_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

SEED       = 42
LABEL_MAP  = {'keluhan': 0, 'saran': 1, 'pujian': 2}
CLASS_NAMES= ['keluhan', 'saran', 'pujian']

# ── FIX 1: NEGASI_PROTECT ─────────────────────────────────────────────────────
NEGASI_PROTECT = {
    'tidak', 'tak', 'bukan', 'belum',
    'tanpa', 'tiada', 'jangan',
    'padahal', 'seharusnya', 'harusnya',
    'malah', 'justru', 'namun', 'tapi', 'tetapi',
    'kurang', 'susah', 'sulit', 'gagal', 'rusak',
    'lambat', 'lemot', 'masalah', 'kendala',
}

# ── FIX 2: STEM_PROTECT ──────────────────────────────────────────────────────
STEM_PROTECT = {
    # KRITIS: rentan distorsi makna saat stemming
    'penangguhan',    # tanpa protect → 'tangguh' (pembalikan makna!)
    'pembayaran', 'pelayanan', 'pendaftaran',
    'penggantian', 'perbaikan', 'keterlambatan',
    'penolakan', 'pembatalan', 'perpanjangan',
    'penonaktifan', 'pengaduan', 'pengaktifan',
    'pelaporan', 'pengembalian', 'pemeriksaan', 'penagihan',
    'pembagian', 'penyaluran', 'pengadaan', 'pelaksanaan',
    # Domain tech
    'bpjs', 'mbg', 'djp', 'coretax', 'spt', 'npwp',
    'login', 'error', 'update', 'upload', 'download',
    'website', 'online', 'offline', 'server', 'sistem',
    'aplikasi', 'faskes', 'puskesmas', 'rujukan',
    'klaim', 'premi', 'iuran', 'pajak', 'akun', 'email',
    'pandawa',
}
STEM_PROTECT.update(NEGASI_PROTECT)

# ── FIX 3: NEGASI_COMBINE ─────────────────────────────────────────────────────
NEGASI_COMBINE = {'tidak', 'tak', 'bukan', 'belum', 'tanpa', 'tiada', 'jangan'}

# Cek semua dictionary tersedia
dict_files = [
    'emoji_dict.csv', 'profanity_dict.csv',
    'slang_dict.csv', 'stopword_dict.csv', 'kbbi_dict.csv'
]
print('Cek dictionary:')
for f in dict_files:
    path = os.path.join(BASE_DIR, f)
    ada  = os.path.exists(path)
    if ada:
        n = len(pd.read_csv(path))
        print(f'  ✅ {f}: {n:,} entri')
    else:
        print(f'  ❌ {f}: TIDAK DITEMUKAN!')

print('\nCek file input:')
if os.path.exists(FILE_IN):
    _df = pd.read_csv(FILE_IN)
    print(f'  ✅ labelled_data_filtered.csv: {len(_df):,} baris')
    del _df
else:
    print(f'  ❌ FILE TIDAK DITEMUKAN!')
print('\n✅ Setup selesai!')

Cek dictionary:
  ✅ emoji_dict.csv: 145 entri
  ✅ profanity_dict.csv: 244 entri
  ✅ slang_dict.csv: 15,131 entri
  ✅ stopword_dict.csv: 759 entri
  ✅ kbbi_dict.csv: 72,423 entri

Cek file input:
  ✅ labelled_data_filtered.csv: 64,349 baris

✅ Setup selesai!


## Cell 2 — Load Semua Dictionary & Definisi Fungsi

In [6]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print('Loading semua dictionary...')
print('='*50)

# ── 1. Stemmer (dengan cache) ─────────────────────────────────────────────────
_factory     = StemmerFactory()
_stemmer_obj = _factory.create_stemmer()

@lru_cache(maxsize=300000)
def stem_word(w):
    if w in STEM_PROTECT: return w
    return _stemmer_obj.stem(w)

# ── 2. Emoji dict → kata deskriptif ─────────────────────────────────────────
# Ganti bagian load emoji di Cell 2 dengan ini:
EMOJI_FILE = os.path.join(BASE_DIR, 'emoji_dict.csv')
emoji_dict = {}

if os.path.exists(EMOJI_FILE):
    _df_em = pd.read_csv(EMOJI_FILE)
    _map   = {'keluhan': 'marah', 'pujian': 'senang', 'saran': 'harap'}

    for _, row in _df_em.iterrows():
        kls  = str(row['klasifikasi']).lower().strip()
        kata = _map.get(kls, '')
        if not kata:
            continue

        # FIX: split multi-emoji yang dipisah koma
        parts = [e.strip() for e in str(row['emoji_list']).split(',')]

        for emoji_char in parts:
            if not emoji_char or emoji_char == 'nan':
                continue
            # FIX: skip entry non-emoji ASCII seperti 'SOS'
            if emoji_char.isascii():
                continue
            # FIX: prioritas pertama jika duplikat (💡 → pujian)
            if emoji_char not in emoji_dict:
                emoji_dict[emoji_char] = kata

# Verifikasi hasil fix
from collections import Counter
dist = Counter(emoji_dict.values())
print(f'✅ Emoji dict fixed: {len(emoji_dict)} emoji individu')
print(f'  marah (keluhan): {dist["marah"]}')
print(f'  senang (pujian): {dist["senang"]}')
print(f'  harap (saran)  : {dist["harap"]}')
# ── 3. Profanity dict → [BADWORD] ────────────────────────────────────────────
# Konsisten dengan pipeline Seprianto v1.0
PROFANITY_FILE = os.path.join(BASE_DIR, 'profanity_dict.csv')
profanity_set  = set()
if os.path.exists(PROFANITY_FILE):
    _df_prof = pd.read_csv(PROFANITY_FILE)
    # Ambil kolom pertama sebagai daftar kata profanity
    profanity_set = set(_df_prof.iloc[:, 0].astype(str).str.lower().str.strip())
print(f'profanity_set  : {len(profanity_set):,} kata kasar → [BADWORD]')

# ── 4. Slang dict ─────────────────────────────────────────────────────────────
SLANG_FILE = os.path.join(BASE_DIR, 'slang_dict.csv')
slang_dict = {}
if os.path.exists(SLANG_FILE):
    _df_s = pd.read_csv(SLANG_FILE)
    slang_raw  = dict(zip(
        _df_s['slang_list'].str.lower().str.strip(),
        _df_s['baku'].str.lower().str.strip()
    ))
    slang_dict = {k: v for k, v in slang_raw.items()
                  if k not in STEM_PROTECT and len(v.split()) <= 3}
print(f'slang_dict     : {len(slang_dict):,} kata slang → baku')

# ── 5. Stopword (dengan NEGASI_PROTECT) ──────────────────────────────────────
_sw_factory   = StopWordRemoverFactory()
stopwords_id  = set(_sw_factory.get_stop_words())
SW_FILE = os.path.join(BASE_DIR, 'stopword_dict.csv')
if os.path.exists(SW_FILE):
    extra_sw = set(pd.read_csv(SW_FILE).iloc[:,0].str.lower().str.strip())
    stopwords_id.update(extra_sw)
# FIX 1: Hapus kata negasi dari stopword!
stopwords_id -= NEGASI_PROTECT
print(f'stopwords_id   : {len(stopwords_id):,} kata (negasi sudah diproteksi)')

# ── 6. KBBI dict ──────────────────────────────────────────────────────────────
KBBI_FILE  = os.path.join(BASE_DIR, 'kbbi_dict.csv')
kbbi_words = set()
USE_KBBI   = False
if os.path.exists(KBBI_FILE):
    kbbi_words = set(pd.read_csv(KBBI_FILE)['kbbi_list'].str.lower().str.strip())
    kbbi_words.update(STEM_PROTECT)
    kbbi_words.update(NEGASI_PROTECT)
    kbbi_words.add('badword')   # token hasil profanity harus lolos filter
    USE_KBBI = True
print(f'kbbi_words     : {len(kbbi_words):,} kata (termasuk negasi, domain, badword)')

print('\n✅ Semua dictionary berhasil dimuat!')
print(f'\nContoh profanity (5 kata): {list(profanity_set)[:5]}')

Loading semua dictionary...
✅ Emoji dict fixed: 226 emoji individu
  marah (keluhan): 55
  senang (pujian): 76
  harap (saran)  : 95
profanity_set  : 213 kata kasar → [BADWORD]
slang_dict     : 14,689 kata slang → baku
stopwords_id   : 799 kata (negasi sudah diproteksi)
kbbi_words     : 72,440 kata (termasuk negasi, domain, badword)

✅ Semua dictionary berhasil dimuat!

Contoh profanity (5 kata): ['cacat', 'lonte', 'goblokkk', 'homo', 'bisyar']


## Cell 3 — Definisi Fungsi Preprocessing v2.0

In [7]:
def negation_handling(tokens):
    """
    FIX 3: Gabungkan kata negasi dengan kata sesudahnya.
    'tidak' + 'aktif' → 'tidak_aktif' (token baru bermakna)
    Token asli 'tidak' tetap disimpan.
    Referensi: Kalaivani et al. (2023)
    """
    result = []
    i = 0
    while i < len(tokens):
        w = tokens[i]
        if (w in NEGASI_COMBINE
                and i + 1 < len(tokens)
                and tokens[i+1] not in NEGASI_COMBINE
                and '_' not in tokens[i+1]):
            result.append(f'{w}_{tokens[i+1]}')  # token gabungan
            result.append(w)                       # kata negasi asli
            i += 2
        else:
            result.append(w)
            i += 1
    return result


def preprocess_v2(text):
    """
    Pipeline preprocessing v2.0 — konsisten dengan Seprianto v1.0
    + perbaikan negasi dan stemming.

    Langkah:
      P1  : Normalisasi newline
      P2  : Hapus URL, mention, hashtag
      P3  : Konversi emoji → kata deskriptif  [emoji_dict.csv]
      P4  : Hapus karakter non-ASCII
      P5  : Hapus tanda baca (kecuali underscore)
      P6  : Case folding
      P7  : Tokenisasi
      P8  : Normalisasi huruf berulang
      P9  : Ganti profanity → badword  [profanity_dict.csv] ← SEBELUMNYA HILANG
      P10 : Koreksi slang → baku  [slang_dict.csv]
      P11 : Stopword removal (NEGASI_PROTECT dipertahankan)  [stopword_dict.csv] ← FIX 1
      P12 : Negation handling → tidak_aktif  ← FIX 3
      P13 : Stemming (STEM_PROTECT dilewati)  ← FIX 2
      P14 : Filter KBBI (token negasi & badword lolos)  [kbbi_dict.csv]
      P15 : Filter panjang ≥ 2 karakter
    """
    def ws(t): return re.sub(r' {2,}', ' ', t).strip()

    # P1: Normalisasi newline
    t = re.sub(r'\r\n|\r|\n', ' ', str(text))
    t = ws(t)

    # P2: Hapus URL, mention, hashtag
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'@\w+', ' ', t)
    t = re.sub(r'#\w+', ' ', t)
    t = ws(t)

    # P3: Konversi emoji → kata deskriptif
    for em, kata in emoji_dict.items():
        t = t.replace(em, f' {kata} ')

    # P4: Hapus karakter non-ASCII (sisa emoji)
    t = re.sub(r'[^\x00-\x7F]+', ' ', t)
    t = ws(t)

    # P5: Hapus tanda baca kecuali underscore
    # Catatan: [BADWORD] → BADWORD (bracket hilang, ini expected)
    t = re.sub(r'[^a-zA-Z0-9\s_]', ' ', t)
    t = ws(t)

    # P6: Case folding
    t = t.lower()

    # P7: Tokenisasi
    toks = t.split()
    if not toks:
        return ''

    # P8: Normalisasi huruf berulang
    toks = [re.sub(r'(.)\1{2,}', r'\1\1', w) for w in toks]

    # P9: Ganti profanity → 'badword' [profanity_dict.csv]
    # Konsisten dengan Seprianto v1.0 ([BADWORD] → setelah P5 → BADWORD → lower → badword)
    toks = ['badword' if w in profanity_set else w for w in toks]

    # P10: Koreksi slang [slang_dict.csv]
    result = []
    for w in toks:
        if w in STEM_PROTECT or w == 'badword':
            result.append(w)
        else:
            result.extend(slang_dict.get(w, w).split())
    toks = result

    # P11: Stopword removal [stopword_dict.csv + NEGASI_PROTECT]
    # Kata negasi tidak dihapus karena stopwords_id -= NEGASI_PROTECT di Cell 2
    toks = [w for w in toks if w not in stopwords_id]

    # P12: Negation handling
    # 'tidak' + 'aktif' → 'tidak_aktif' + 'tidak'
    toks = negation_handling(toks)

    # P13: Stemming [Sastrawi + STEM_PROTECT]
    # Token gabungan (ada underscore) dan STEM_PROTECT tidak di-stem
    toks = [
        w if ('_' in w or w == 'badword') else stem_word(w)
        for w in toks
    ]

    # P14: Filter KBBI [kbbi_dict.csv]
    if USE_KBBI:
        toks = [
            w for w in toks
            if '_' in w          # token negasi gabungan → lolos
            or w == 'badword'    # token profanity → lolos
            or len(w) <= 2       # kata pendek → lolos
            or w in kbbi_words   # ada di KBBI → lolos
        ]

    # P15: Filter panjang
    toks = [w for w in toks if len(w) >= 2]

    return ' '.join(toks)


print('Fungsi preprocess_v2 siap!')
print()
print('Pipeline lengkap (15 langkah):')
steps = [
    ('P1',  'Normalisasi newline'),
    ('P2',  'Hapus URL, mention, hashtag'),
    ('P3',  'Emoji → kata deskriptif  [emoji_dict.csv]'),
    ('P4',  'Hapus non-ASCII'),
    ('P5',  'Hapus tanda baca'),
    ('P6',  'Case folding'),
    ('P7',  'Tokenisasi'),
    ('P8',  'Normalisasi huruf berulang'),
    ('P9',  'Profanity → badword  [profanity_dict.csv]  ★ v1.0 konsisten'),
    ('P10', 'Slang → baku  [slang_dict.csv]'),
    ('P11', 'Stopword removal  [stopword_dict.csv]  ★ FIX 1: negasi diproteksi'),
    ('P12', 'Negation handling  ★ FIX 3: tidak_aktif'),
    ('P13', 'Stemming  ★ FIX 2: STEM_PROTECT'),
    ('P14', 'Filter KBBI  [kbbi_dict.csv]'),
    ('P15', 'Filter panjang ≥ 2 karakter'),
]
for kode, desc in steps:
    print(f'  {kode:4s}: {desc}')

Fungsi preprocess_v2 siap!

Pipeline lengkap (15 langkah):
  P1  : Normalisasi newline
  P2  : Hapus URL, mention, hashtag
  P3  : Emoji → kata deskriptif  [emoji_dict.csv]
  P4  : Hapus non-ASCII
  P5  : Hapus tanda baca
  P6  : Case folding
  P7  : Tokenisasi
  P8  : Normalisasi huruf berulang
  P9  : Profanity → badword  [profanity_dict.csv]  ★ v1.0 konsisten
  P10 : Slang → baku  [slang_dict.csv]
  P11 : Stopword removal  [stopword_dict.csv]  ★ FIX 1: negasi diproteksi
  P12 : Negation handling  ★ FIX 3: tidak_aktif
  P13 : Stemming  ★ FIX 2: STEM_PROTECT
  P14 : Filter KBBI  [kbbi_dict.csv]
  P15 : Filter panjang ≥ 2 karakter


## Cell 4 — Demo & Verifikasi Pipeline

In [8]:
test_cases = [
    ('kenapa sih bpjs saya tidak aktif? padahal saya sudah bayar tiap bulan',
     'keluhan', 'Kasus Negasi 1'),
    ('hampir sebulan belum aktif juga katanya penangguhan pembayaran padahal udah selesaikan administrasinya',
     'keluhan', 'Kasus Negasi 2 + Distorsi Stem'),
    ('bpjs bagus banget pelayanannya terima kasih banyak',
     'pujian', 'Kasus Pujian Normal'),
    ('sebaiknya coretax diperbaiki dulu sebelum diluncurkan',
     'saran', 'Kasus Saran'),
    ('dasar bajingan bpjs gak bener kerja',
     'keluhan', 'Kasus Profanity'),
]

print('='*65)
print('DEMO PIPELINE v2.0')
print('='*65)
for teks, label, nama in test_cases:
    hasil = preprocess_v2(teks)
    toks  = hasil.split()
    negasi_toks  = [t for t in toks if '_' in t]
    badword_toks = [t for t in toks if t == 'badword']
    print(f'\n[{nama}] label={label}')
    print(f'  Input  : {teks}')
    print(f'  Output : {hasil}')
    if negasi_toks:
        print(f'  ★ Token negasi  : {negasi_toks}')
    if badword_toks:
        print(f'  ★ Token profanity: {badword_toks}')

# Verifikasi khusus
print('\n' + '='*65)
print('VERIFIKASI KRITIS')
print('='*65)

# 1. Cek tidak dihapus
out1 = preprocess_v2('bpjs tidak aktif')
cek1 = 'tidak' in out1.split()
print(f'\n1. Kata negasi dipertahankan: {cek1}')
print(f'   Input: "bpjs tidak aktif" → Output: "{out1}"')

# 2. Cek penangguhan tidak berubah menjadi tangguh
out2 = preprocess_v2('ada penangguhan pembayaran')
cek2 = 'tangguh' not in out2.split()
print(f'\n2. Penangguhan tidak → tangguh: {cek2}')
print(f'   Input: "ada penangguhan pembayaran" → Output: "{out2}"')

# 3. Cek token tidak_aktif muncul
out3 = preprocess_v2('bpjs tidak aktif')
cek3 = 'tidak_aktif' in out3.split()
print(f'\n3. Token tidak_aktif muncul: {cek3}')
print(f'   Input: "bpjs tidak aktif" → Output: "{out3}"')

# 4. Cek profanity menjadi badword
# Ambil 1 kata dari profanity_set untuk test
if profanity_set:
    sample_prof = list(profanity_set)[0]
    out4 = preprocess_v2(f'bpjs {sample_prof} sekali')
    cek4 = 'badword' in out4.split()
    print(f'\n4. Profanity → badword: {cek4}')
    print(f'   Input: "bpjs {sample_prof} sekali" → Output: "{out4}"')

semua_ok = all([cek1, cek2, cek3])
print(f'\n{"✅ Semua verifikasi PASSED!" if semua_ok else "⚠️  Ada verifikasi FAILED, cek kembali!"}')

DEMO PIPELINE v2.0

[Kasus Negasi 1] label=keluhan
  Input  : kenapa sih bpjs saya tidak aktif? padahal saya sudah bayar tiap bulan
  Output : sih bpjs tidak_aktif tidak padahal bayar
  ★ Token negasi  : ['tidak_aktif']

[Kasus Negasi 2 + Distorsi Stem] label=keluhan
  Input  : hampir sebulan belum aktif juga katanya penangguhan pembayaran padahal udah selesaikan administrasinya
  Output : bulan belum_aktif belum penangguhan pembayaran padahal selesai administrasi
  ★ Token negasi  : ['belum_aktif']

[Kasus Pujian Normal] label=pujian
  Input  : bpjs bagus banget pelayanannya terima kasih banyak
  Output : bpjs bagus banget layan terima kasih

[Kasus Saran] label=saran
  Input  : sebaiknya coretax diperbaiki dulu sebelum diluncurkan
  Output : coretax baik luncur

[Kasus Profanity] label=keluhan
  Input  : dasar bajingan bpjs gak bener kerja
  Output : dasar badword bpjs tidak
  ★ Token profanity: ['badword']

VERIFIKASI KRITIS

1. Kata negasi dipertahankan: True
   Input: "bpjs tidak 

## Cell 5 — Preprocessing `labelled_data_filtered.csv`
> Estimasi waktu: **5–20 menit** (tergantung CPU)
> Stemming Sastrawi lebih lambat dari v1.0 karena lebih banyak token yang diproses


In [9]:
print(f'Loading {FILE_IN}...')
df = pd.read_csv(FILE_IN)
df = df.dropna(subset=['text', 'label_pks']).reset_index(drop=True)

print(f'Total baris    : {len(df):,}')
print(f'\nDistribusi label:')
for lbl, cnt in df['label_pks'].value_counts().items():
    pct = cnt/len(df)*100
    bar = chr(9608)*int(pct/3)
    print(f'  {lbl:10s}: {cnt:6,} ({pct:.1f}%) {bar}')

print(f'\nDistribusi confidence:')
print(f'  Manual (conf=1.0)  : {(df.confidence==1.0).sum():,}')
print(f'  Otomatis (conf<1.0): {(df.confidence<1.0).sum():,}')

# Preprocessing
print(f'\nPreprocessing {len(df):,} baris...')
print('(Estimasi 5-20 menit, harap tunggu...)')
t0 = time.time()
df['text_v2'] = df['text'].fillna('').apply(preprocess_v2)
elapsed = time.time() - t0

# Statistik
n_empty     = (df['text_v2'].str.strip() == '').sum()
avg_tok     = df['text_v2'].apply(lambda t: len(t.split())).mean()
n_negasi    = df['text_v2'].apply(lambda t: sum(1 for w in t.split() if '_' in w)).sum()
n_badword   = df['text_v2'].apply(lambda t: t.split().count('badword')).sum()

print(f'\nHasil preprocessing v2.0:')
print(f'  Waktu           : {elapsed:.0f}s ({elapsed/len(df)*1000:.1f}ms/baris)')
print(f'  Teks kosong     : {n_empty} baris (dihapus)')
print(f'  Avg token/teks  : {avg_tok:.1f}')
print(f'  Token tidak_X   : {n_negasi:,} (negation handling)')
print(f'  Token badword   : {n_badword:,} (profanity replaced)')

# Hapus baris kosong
df_clean = df[df['text_v2'].str.strip() != ''].copy()
print(f'  Dataset bersih  : {len(df_clean):,} baris')

# Simpan
path_full = os.path.join(PREP_DIR, 'data_preprocessed_v2.csv')
df_clean.to_csv(path_full, index=False, encoding='utf-8-sig')
print(f'\n✅ Tersimpan: {path_full}')

Loading C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\labelled_data_filtered.csv...
Total baris    : 64,349

Distribusi label:
  keluhan   : 41,267 (64.1%) █████████████████████
  pujian    : 14,413 (22.4%) ███████
  saran     :  8,669 (13.5%) ████

Distribusi confidence:
  Manual (conf=1.0)  : 18,534
  Otomatis (conf<1.0): 45,815

Preprocessing 64,349 baris...
(Estimasi 5-20 menit, harap tunggu...)

Hasil preprocessing v2.0:
  Waktu           : 23s (0.4ms/baris)
  Teks kosong     : 4485 baris (dihapus)
  Avg token/teks  : 7.0
  Token tidak_X   : 27,071 (negation handling)
  Token badword   : 4,332 (profanity replaced)
  Dataset bersih  : 59,864 baris

✅ Tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\prep_v2\data_preprocessed_v2.csv


In [10]:
# ── DIAGNOSTIK LENGKAP ─────────────────────────────────────────────────────
print("="*60)
print("DIAGNOSTIK PREPROCESSING v2.0")
print("="*60)

# 1. Cek stemmer benar-benar dipanggil
print("\n1. Test stemmer (harus lambat jika cold, cepat jika cached):")
test_stem = [
    ("pelayanan",    "layan"),         # harus di-stem
    ("penangguhan",  "penangguhan"),   # STEM_PROTECT → tidak di-stem
    ("memperbaiki",  "baik"),          # harus di-stem
    ("bagus",        "bagus"),         # tidak berubah
    ("pembayaran",   "pembayaran"),    # STEM_PROTECT → tidak di-stem
]
for kata, expected in test_stem:
    hasil  = stem_word(kata)
    status = "✅" if hasil == expected else f"❌ dapat '{hasil}', harusnya '{expected}'"
    print(f"   stem_word('{kata}') → '{hasil}'  {status}")

# 2. Cek lru_cache info
cache_info = stem_word.cache_info()
print(f"\n2. lru_cache status:")
print(f"   Hits  : {cache_info.hits:,}   (panggilan yang dikembalikan dari cache)")
print(f"   Misses: {cache_info.misses:,}  (panggilan yang benar-benar ke Sastrawi)")
print(f"   Cache size: {cache_info.currsize:,} kata unik tersimpan")
if cache_info.misses > 0:
    hit_rate = cache_info.hits / (cache_info.hits + cache_info.misses) * 100
    print(f"   Hit rate: {hit_rate:.1f}% ← ini yang membuat cepat!")

# 3. Cek sample output dari CSV
print("\n3. Sample output dari data_preprocessed_v2.csv:")
df_check = pd.read_csv(os.path.join(PREP_DIR, 'data_preprocessed_v2.csv'))

# Cari baris mengandung kata negasi
mask_neg = df_check['text'].str.contains(
    r'\btidak\b|\bbelum\b|\bbukan\b', na=False, regex=True
)
print(f"   Baris dengan kata negasi (total): {mask_neg.sum():,}")
print(f"\n   Contoh 5 baris dengan negasi:")
for _, row in df_check[mask_neg].head(5).iterrows():
    has_neg_tok = '_' in str(row['text_v2'])
    print(f"   [{row['label_pks']}]")
    print(f"     Asli   : '{str(row['text'])[:70]}'")
    print(f"     v2.0   : '{str(row['text_v2'])[:70]}'")
    print(f"     Token _ : {'✅ ADA' if has_neg_tok else '❌ TIDAK ADA'}")
    print()

# 4. Cek penangguhan tidak berubah jadi tangguh
mask_pngg = df_check['text'].str.contains('penangguhan', na=False)
print(f"4. Baris dengan kata 'penangguhan': {mask_pngg.sum()}")
if mask_pngg.sum() > 0:
    for _, row in df_check[mask_pngg].head(3).iterrows():
        v2_teks = str(row['text_v2'])
        ada_tangguh = 'tangguh' in v2_teks.split()
        ada_pngg    = 'penangguhan' in v2_teks.split()
        print(f"   Asli  : '{str(row['text'])[:60]}'")
        print(f"   v2.0  : '{v2_teks[:60]}'")
        print(f"   'tangguh' muncul : {ada_tangguh}  ← {'❌ STEM PROTECT GAGAL!' if ada_tangguh else '✅ Aman'}")
        print(f"   'penangguhan' ada: {ada_pngg}")
        print()

# 5. Cek badword
print(f"5. Token badword:")
mask_bw = df_check['text_v2'].str.contains('badword', na=False)
print(f"   Baris dengan 'badword': {mask_bw.sum():,}")
if mask_bw.sum() > 0:
    for _, row in df_check[mask_bw].head(3).iterrows():
        print(f"   Asli : '{str(row['text'])[:60]}'")
        print(f"   v2.0 : '{str(row['text_v2'])[:60]}'")
        print()

print("="*60)
print("Analisis kecepatan:")
print(f"   13 detik untuk {len(df_check):,} baris = normal jika:")
print(f"   - lru_cache hit rate tinggi (banyak kata berulang)")
print(f"   - Banyak token difilter sebelum sampai ke Sastrawi")
print(f"   - Dataset media sosial punya vocabulary terbatas")

DIAGNOSTIK PREPROCESSING v2.0

1. Test stemmer (harus lambat jika cold, cepat jika cached):
   stem_word('pelayanan') → 'pelayanan'  ❌ dapat 'pelayanan', harusnya 'layan'
   stem_word('penangguhan') → 'penangguhan'  ✅
   stem_word('memperbaiki') → 'baik'  ✅
   stem_word('bagus') → 'bagus'  ✅
   stem_word('pembayaran') → 'pembayaran'  ✅

2. lru_cache status:
   Hits  : 446,323   (panggilan yang dikembalikan dari cache)
   Misses: 30,577  (panggilan yang benar-benar ke Sastrawi)
   Cache size: 30,577 kata unik tersimpan
   Hit rate: 93.6% ← ini yang membuat cepat!

3. Sample output dari data_preprocessed_v2.csv:
   Baris dengan kata negasi (total): 5,575

   Contoh 5 baris dengan negasi:
   [keluhan]
     Asli   : 'Hr ini / dr pagi sampai slrg : tidak bisa login d coretax knp ya min?'
     v2.0   : 'pagi tidak_login tidak coretax admin'
     Token _ : ✅ ADA

   [keluhan]
     Asli   : 'Coretax tidak Bisa LOG In....padahal User Password sudah benar dan ses'
     v2.0   : 'coretax tidak_lo